# M1M3 z-Gradient Survey (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-07
**Last Modified:** 2026-07-07
**Status:** In Progress
**Keywords:** M1M3, thermal gradient, z_gradient, EFD, thermocouples, bounce test

## Description

Survey the M1M3 vertical thermal gradient (`z_gradient`) across all `science` and
`acq` exposures over a date range.  `z_gradient` is **not** in the ConsDB — it is
derived from the EFD M1M3 thermocouple telemetry via
`lsst.ts.m1m3.utils.ThermocoupleAnalysis` (the same `xyz_r_gradients` the AOS
`ts_intrinsic_wavefront` pipeline uses to enrich its visit tables).

For speed over a multi-month range the EFD is loaded in **multi-day chunks** at a
**coarse cadence** (default 5 min — z_gradient varies on ~10-min scales), and the
gradient time series is interpolated to each exposure's start time.  Loading
per-night at 30 s (the pipeline default) is ~100 EFD round-trips and is far slower.

Key functionality:
1. List all `science`/`acq` exposures over [START, END] from the Butler (one query).
2. Load M1M3 `z_gradient` from the EFD in `CHUNK_DAYS` windows at `TIME_BIN_SEC`
   cadence (tqdm per-chunk), interpolate to each exposure.
3. Cache to parquet; plot z_gradient vs ordinal day_obs (scatter + nightly median)
   and its per-night distribution (violin).

**Output:** `z_gradient_science_acq.parquet` + a scatter/violin figure.

**References:** `ts_intrinsic_wavefront/.../intrinsics_lib.py` (`get_m1m3_data`,
`ThermocoupleAnalysis.xyz_r_gradients`); ConsDB `cdb_lsstcam` schema (z_gradient absent).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-07 | Aaron Roodman | Initial version — science/acq z_gradient survey, chunked/coarse EFD loads, scatter/violin |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Fetch z_gradient per exposure](#fetch)
4. [Plots](#plots)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — all configurable values collected here
# ============================================================
from datetime import date

START = "2026-03-15"                 # first calendar day (inclusive)
END = date.today().isoformat()       # last calendar day (inclusive) — default: today
KEEP_TYPES = ("science", "acq")      # exposure.observation_type values to keep
BUTLER_REPO = "/repo/main"
EFD_NAME = "usdf_efd"                 # EFD instance for this site
CHUNK_DAYS = 7                        # EFD load window (bigger = fewer round-trips)
TIME_BIN_SEC = 300                   # EFD gradient cadence (s); z_gradient is slow
OUT = "z_gradient_science_acq.parquet"   # cache path (cwd); set elsewhere if desired
FIG = "z_gradient_science_acq.png"

<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm                    # plain text bar (no ipywidgets dependency)
from astropy.time import Time

from lsst.daf.butler import Butler
from lsst_efd_client import EfdClient
from lsst.ts.m1m3.utils import ThermocoupleAnalysis

butler = Butler(BUTLER_REPO, instrument="LSSTCam")
# Construct EfdClient directly with output_mode="dataframe"; makeEfdClient() can fail
# with "Invalid output format" on some summit_utils / lsst_efd_client combinations.
efd = EfdClient(EFD_NAME, output_mode="dataframe")

<a id='fetch'></a>
## Fetch z_gradient per exposure

EFD loaded in multi-day chunks at coarse cadence (tune `CHUNK_DAYS` / `TIME_BIN_SEC`); `await` works at notebook top level. In a plain script wrap the loop in `async def` and use `asyncio.run(...)`.

In [ ]:
# 1) all science/acq exposures in the range, in ONE Butler query
start_day = int(START.replace("-", "")); end_day = int(END.replace("-", ""))
recs = butler.registry.queryDimensionRecords(
    "exposure", instrument="LSSTCam",
    where=f"exposure.day_obs >= {start_day} AND exposure.day_obs <= {end_day}")
exp = [(r.day_obs, r.seq_num, r.observation_type, r.timespan.begin)
       for r in recs if r.observation_type in KEEP_TYPES]
df = pd.DataFrame(exp, columns=["day_obs", "seq_num", "observation_type", "t_begin"])
df = df.sort_values("t_begin").reset_index(drop=True)
df["t_ns"] = (Time(list(df["t_begin"])).unix * 1e9).astype("int64")   # UTC ns
df["z_gradient"] = np.nan
print(f"{len(df)} science/acq exposures over {df.day_obs.nunique()} nights")

# 2) EFD M1M3 gradients in CHUNK_DAYS windows at TIME_BIN_SEC cadence -> interp to exposures
edges = pd.date_range(START, pd.Timestamp(END) + pd.Timedelta(days=1),
                      freq=f"{CHUNK_DAYS}D")
for i in tqdm(range(len(edges) - 1), desc=f"{CHUNK_DAYS}-day EFD chunks"):
    c0, c1 = edges[i], edges[i + 1]
    sel = (df.t_ns >= c0.value) & (df.t_ns < c1.value)
    if not sel.any():
        continue
    try:
        tca = ThermocoupleAnalysis(efd)
        await tca.load(Time(c0.isoformat(), scale="utc"),
                       Time(c1.isoformat(), scale="utc"), time_bin=TIME_BIN_SEC)
        g = tca.xyz_r_gradients
        gt = pd.to_datetime(g.index, utc=True).astype("int64").to_numpy()
        gz = np.asarray(g["z_gradient"], float)
        o = np.argsort(gt)
        df.loc[sel, "z_gradient"] = np.interp(df.loc[sel, "t_ns"].to_numpy(),
                                              gt[o], gz[o])
    except Exception as e:
        tqdm.write(f"  {c0.date()}..{c1.date()}: failed ({type(e).__name__}: {e})")

df.drop(columns=["t_begin", "t_ns"]).to_parquet(OUT)
print(f"z_gradient filled for {int(np.isfinite(df.z_gradient).sum())}/{len(df)} "
      f"exposures -> {OUT}")

<a id='plots'></a>
## Plots

In [ ]:
df = pd.read_parquet(OUT)
df = df[np.isfinite(df.z_gradient)]
days = np.sort(df.day_obs.unique())
ordmap = {d: i for i, d in enumerate(days)}          # ordinal night index (gap-free)
df["day_ord"] = df.day_obs.map(ordmap)
ticks = np.arange(0, len(days), max(1, len(days) // 15))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(max(10, 0.22 * len(days)), 9),
                             constrained_layout=True)

# scatter: one point per exposure + nightly median
a1.scatter(df.day_ord, df.z_gradient, s=6, alpha=0.25, color="steelblue")
med = df.groupby("day_ord").z_gradient.median()
a1.plot(med.index, med.values, "-o", ms=3, color="crimson", lw=0.8, label="nightly median")
a1.set_ylabel("M1M3 z_gradient"); a1.legend(fontsize=8); a1.grid(alpha=0.3)
a1.set_title(f"z_gradient per science/acq exposure  "
             f"({df.day_obs.min()}-{df.day_obs.max()}, N={len(df)})")
a1.set_xticks(ticks); a1.set_xticklabels([str(days[i]) for i in ticks], rotation=90, fontsize=7)

# violin: distribution per night
data = []
for i in range(len(days)):
    d = df.z_gradient[df.day_ord == i].values
    data.append(d if d.size >= 2 else (np.full(2, d[0]) if d.size == 1
                                       else np.array([np.nan, np.nan])))
vp = a2.violinplot(data, positions=np.arange(len(days)), widths=0.9,
                   showmedians=True, showextrema=False)
for b in vp["bodies"]:
    b.set_facecolor("steelblue"); b.set_alpha(0.5)
a2.set_ylabel("M1M3 z_gradient"); a2.set_xlabel("day_obs"); a2.grid(axis="y", alpha=0.3)
a2.set_xticks(ticks); a2.set_xticklabels([str(days[i]) for i in ticks], rotation=90, fontsize=7)

fig.savefig(FIG, dpi=140)
plt.show()